# Introduction to Convolutional Neural Networks (CNNs)

This notebook explains CNNs from first principles. It follows the same learning style as the ANN notebook: concept, intuition, example, key points, and caution points.

## Learning Goals

- Understand images as numerical data.
- Understand why MLPs struggle with images.
- Learn convolution, kernels, padding, stride, channels, pooling, and feature maps.
- Understand standard CNN architecture.
- Learn common CNN tasks such as classification, detection, and segmentation at a conceptual level.
- Understand CNN training, evaluation, visualization, and common mistakes.

> **Scope:** This is a theory notebook. Code is kept in `02_Introduction_To_CNNs_add_code.ipynb`.


# Image as Data

Machines do not see images as pictures. They see images as arrays of numbers.

## Pixels

A pixel is the smallest unit of a digital image. Each pixel stores intensity information.

## Types of Images

| Image Type | Shape Example | Meaning |
|---|---:|---|
| Binary image | $H \times W$ | Each pixel is usually 0 or 1 |
| Grayscale image | $H \times W$ | Each pixel is usually 0 to 255 |
| RGB image | $H \times W \times 3$ | Each pixel has red, green, and blue values |

## Example

A grayscale Fashion-MNIST image has shape:

$$28 \times 28$$

That means it has 784 pixel values.

An RGB image of size 224 by 224 has shape:

$$224 \times 224 \times 3$$

That means:

- 224 rows
- 224 columns
- 3 color channels

## Key Points

- Images have spatial structure.
- Nearby pixels are usually related.
- Channels store different types of visual information.
- CNNs are designed to preserve and exploit this structure.

## Caution Points

- Flattening an image too early destroys spatial relationships.
- Pixel values should usually be normalized before training.
- Image shape conventions may differ: some libraries use channels last, while PyTorch uses channels first.


# Why MLPs Struggle with Images

An MLP expects a vector. To use an MLP for images, we flatten the image.

Example:

$$28 \times 28 \rightarrow 784$$

This works mathematically, but it is not ideal.

## Problem 1: Loss of Spatial Structure

When an image is flattened, pixels that were nearby in the image may become far apart in the vector.

Example:

In an image, an eye, nose, and mouth have spatial arrangement. Flattening does not explicitly preserve that arrangement.

## Problem 2: Too Many Parameters

Suppose an RGB image has shape:

$$224 \times 224 \times 3 = 150,528$$

If the first hidden layer has 1,000 neurons:

$$150,528 \times 1,000 = 150,528,000$$

That is over 150 million weights in the first layer alone.

## Problem 3: Poor Translation Handling

An MLP may learn that a pattern matters at one exact position. But in images, the same object can appear in different locations.

Example:

A shoe should still be recognized as a shoe whether it appears slightly left or right.

## Key Points

- MLPs can process images, but they are inefficient for vision.
- MLPs do not naturally exploit local patterns.
- MLPs do not share weights across image locations.
- CNNs solve these problems using local receptive fields and weight sharing.

## Caution Points

- Do not assume more dense layers will automatically solve image problems.
- Parameter explosion increases overfitting risk.
- Vision models should respect image geometry whenever possible.


# The Convolution Operation

Convolution is the core operation of a CNN.

A small matrix called a **kernel** or **filter** slides over the image and computes local weighted sums.

## Kernel Intuition

A kernel is like a small pattern detector.

Examples:

- Edge detector
- Blur filter
- Sharpen filter
- Texture detector

## Simple 3 by 3 Kernel

$$
\begin{bmatrix}
-1 & -1 & -1 \\
0 & 0 & 0 \\
1 & 1 & 1
\end{bmatrix}
$$

This type of kernel can respond strongly to horizontal edges.

## How Convolution Works

1. Place the kernel over a small region of the image.
2. Multiply overlapping values.
3. Add the results.
4. Move the kernel to the next location.
5. Repeat across the image.

The output is called a **feature map**.

## Receptive Field

The receptive field is the region of the input image that affects one output value.

In early layers, receptive fields are small. In deeper layers, neurons indirectly see larger regions of the image.

## Key Points

- Convolution detects local patterns.
- The same kernel is applied across many positions.
- The output feature map shows where a pattern appears.
- Early CNN filters often learn edges and textures.

## Caution Points

- A kernel does not understand the whole image at once.
- Kernel size affects how much local context is captured.
- Too large a kernel can increase computation and parameters.


# Channels, Filters, and Feature Maps

CNNs usually process multiple channels.

## Input Channels

Examples:

- Grayscale image: 1 channel
- RGB image: 3 channels
- A hidden CNN layer: many learned channels

## Filters

A filter produces one output feature map.

If a convolutional layer has 32 filters, it produces 32 feature maps.

## Example

Input:

$$28 \times 28 \times 1$$

Convolution with 32 filters:

$$28 \times 28 \times 32$$

Each output channel represents a different learned pattern.

## Parameter Count

For a convolution layer:

$$\text{parameters} = (k_h \times k_w \times C_{in} \times C_{out}) + C_{out}$$

where:

- $k_h, k_w$ are kernel height and width
- $C_{in}$ is number of input channels
- $C_{out}$ is number of filters
- $C_{out}$ bias terms are added if bias is used

## Example

A 3 by 3 convolution with 3 input channels and 64 filters:

$$3 \times 3 \times 3 \times 64 + 64 = 1,792$$

This is far smaller than a dense layer on a flattened large image.

## Key Points

- More filters mean more learned pattern detectors.
- Channel count usually increases in deeper CNN layers.
- Spatial size often decreases while channel depth increases.

## Caution Points

- More filters increase computation and memory.
- Too few filters may underfit.
- Too many filters can overfit small datasets.


# Padding and Stride

Padding and stride control the spatial size of CNN outputs.

## Stride

Stride is how far the kernel moves each step.

- Stride 1: move one pixel at a time.
- Stride 2: move two pixels at a time.

Larger stride reduces output size.

## Padding

Padding adds extra pixels around the border of the image.

Why use padding?

- Preserve spatial size.
- Allow border pixels to influence the output more fairly.
- Control shrinking across layers.

## Output Size Formula

For one spatial dimension:

$$
O = \left\lfloor \frac{I + 2P - K}{S} \right\rfloor + 1
$$

where:

- $I$ = input size
- $P$ = padding
- $K$ = kernel size
- $S$ = stride
- $O$ = output size

## Example

Input size 28, kernel size 3, padding 1, stride 1:

$$
O = \frac{28 + 2(1) - 3}{1} + 1 = 28
$$

So the output spatial size remains 28.

## Key Points

- Padding preserves border information.
- Stride downsamples spatial dimensions.
- Padding and stride strongly affect architecture design.

## Caution Points

- Repeated shrinking can remove too much spatial information.
- Incorrect output-size calculations can break the classifier head.
- Padding does not create new real information; it only changes boundary handling.


# Pooling Layers

Pooling reduces spatial size while keeping important information.

## Max Pooling

Max pooling takes the maximum value in each local region.

Example:

$$
\begin{bmatrix}
1 & 3 \\
2 & 4
\end{bmatrix}
\rightarrow 4
$$

Max pooling keeps the strongest activation.

## Average Pooling

Average pooling takes the average value in each local region.

It produces smoother summaries than max pooling.

## Why Pooling Helps

- Reduces computation.
- Reduces memory use.
- Makes features more tolerant to small shifts.
- Helps build compact representations.

## CNN Size Walkthrough

For Fashion-MNIST:

$$28 \times 28 \rightarrow 14 \times 14 \rightarrow 7 \times 7 \rightarrow 3 \times 3$$

Spatial dimensions shrink, while channel count usually grows.

## Key Points

- Pooling does not learn weights.
- Pooling compresses spatial information.
- Max pooling is common in simple CNNs.

## Caution Points

- Too much pooling can destroy useful location information.
- Segmentation tasks need spatial detail, so excessive pooling is risky.
- Pooling is not always required; strided convolution can also downsample.


# Standard CNN Architecture

A common CNN for image classification has two parts:

1. Feature extractor
2. Classifier

## Feature Extractor

The feature extractor uses convolution blocks.

Common block:

$$\text{Conv} \rightarrow \text{ReLU} \rightarrow \text{Pool}$$

Example:

$$1 \rightarrow 32 \rightarrow 64 \rightarrow 128$$

The spatial size decreases while the channel count increases.

## Classifier

After feature extraction, the feature maps are flattened or globally pooled and passed to dense layers.

Example:

$$128 \times 3 \times 3 \rightarrow 1152 \rightarrow 10$$

For Fashion-MNIST, the final output has 10 classes.

## Full Flow

Input image:

$$1 \times 28 \times 28$$

CNN feature extractor:

$$32 \times 14 \times 14 \rightarrow 64 \times 7 \times 7 \rightarrow 128 \times 3 \times 3$$

Classifier:

$$1152 \rightarrow 10$$

## Key Points

- Convolution blocks learn visual features.
- Classifier layers make the final decision.
- Early layers learn simple patterns.
- Deeper layers learn more task-specific patterns.

## Caution Points

- Always track tensor shapes through the network.
- The flatten size must match the classifier input.
- A CNN can still overfit if it is too large for the dataset.


# Common Computer Vision Tasks

CNNs are used in many vision tasks.

## Image Classification

Goal:

Assign one label to the entire image.

Example:

The image is a shirt, shoe, cat, or dog.

## Object Detection

Goal:

Find objects and draw bounding boxes around them.

Example:

Detect all cars and pedestrians in an image.

## Semantic Segmentation

Goal:

Assign a class label to every pixel.

Example:

Mark which pixels are road, sky, car, and person.

## Instance Segmentation

Goal:

Separate individual object instances pixel by pixel.

Example:

Identify each separate person in a crowd.

## Key Points

- Classification predicts one label per image.
- Detection predicts object locations and classes.
- Segmentation predicts labels at pixel level.
- CNN feature extraction is useful across these tasks.

## Caution Points

- Do not confuse classification with detection.
- Segmentation needs spatial resolution, not just class prediction.
- Different tasks need different output formats and loss functions.


# Training and Evaluation of CNNs

CNN training follows the same deep learning loop as ANNs.

## Training Flow

1. Prepare image data.
2. Normalize images.
3. Build the CNN.
4. Perform forward pass.
5. Compute loss.
6. Backpropagate gradients.
7. Update parameters.
8. Evaluate on validation data.

## Common Loss Functions

| Task | Output | Loss |
|---|---|---|
| Binary image classification | 1 probability | Binary cross-entropy |
| Multiclass classification | Class scores | Cross-entropy |
| Segmentation | Pixel-wise class scores | Pixel-wise cross-entropy / Dice loss |

## Evaluation Metrics

For classification:

- Accuracy
- Precision
- Recall
- F1-score
- Confusion matrix

For segmentation:

- Pixel accuracy
- Intersection over Union
- Dice score

## Key Points

- CNNs are trained with backpropagation.
- Data normalization matters.
- Validation performance matters more than training accuracy.
- Use task-specific metrics.

## Caution Points

- High training accuracy with low validation accuracy means overfitting.
- Data leakage can make results look falsely good.
- Class imbalance can make accuracy misleading.


# Visualizing CNNs

CNNs are often treated as black boxes, but we can inspect what they learn.

## Filter Visualization

Early filters may look like edge, corner, or texture detectors.

## Feature Maps

Feature maps show where a filter activates strongly.

Example:

If a filter detects vertical edges, its feature map lights up around vertical edges in the image.

## Class Activation Ideas

Some visualization methods highlight which image regions influenced the final prediction most strongly.

## Why Visualization Helps

- Understand learned patterns.
- Debug unexpected behavior.
- Detect whether the model focuses on the right object.
- Explain model decisions more clearly.

## Caution Points

- Visualizations are interpretations, not perfect explanations.
- A model may use shortcuts or background cues.
- Good-looking feature maps do not guarantee good generalization.


# Common CNN Mistakes and Practical Checklist

## Common Mistakes

- Flattening images before convolution.
- Forgetting channel order.
- Using the wrong input shape.
- Making the classifier input size incorrect.
- Using too much pooling.
- Training on unnormalized images.
- Judging only by training accuracy.
- Ignoring class imbalance.

## Practical Checklist

Before training:

- Check image shapes.
- Check label format.
- Normalize inputs.
- Confirm output layer matches number of classes.
- Confirm loss function matches the task.

During training:

- Track training loss.
- Track validation loss.
- Watch for overfitting.
- Save the best model based on validation performance.

After training:

- Inspect confusion matrix.
- Look at wrong predictions.
- Visualize feature maps if needed.

## Final Key Point

CNNs are powerful because they combine three ideas:

1. Local connectivity
2. Weight sharing
3. Hierarchical feature learning
